In [1]:
spark

In [3]:
# Import SparkSession
from pyspark. sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder \
.appName("Handling Dates in PySpark") \
.getOrCreate()

25/12/28 18:09:22 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [6]:
# Sending the Data To HDFS Via Notebook

# Create a CSV file
csv_data = """id, date_iso, date_dmy, date_mdy, timestamp
1,2023-01-15,15/01/2023,01/15/2023,2023-01-15 10:30:00
2,2023-05-20,20/05/2023,05/20/2023,2023-05-20 15:45:00
3,InvalidDate, 31/02/2023,02/31/2023, InvalidTimestamp
4,, ..
"""
# Save the CSV file
with open("dates_data.csv", "w") as f:
      f.write(csv_data)

In [7]:
! ls *dates_data*

dates_data.csv


In [8]:
! hadoop fs -ls

Found 2 items
drwxr-xr-x   - root hadoop          0 2025-12-28 18:08 .sparkStaging
drwxr-xr-x   - root hadoop          0 2025-12-20 18:46 data


In [10]:
! hadoop fs -put dates_data.csv /data/dates_data.csv

In [12]:
! hadoop fs -ls /data/dates_data.csv

-rw-r--r--   2 root hadoop        216 2025-12-28 18:11 /data/dates_data.csv


In [14]:
# Sample data with multiple date formats
data = [
(1, "2023-01-15", "15/01/2023", "01/15/2023", "2023-01-15 10:30:00"),
(2, "2023-05-20", "20/05/2023", "05/20/2023", "2023-05-20 15:45:00"),
(3, "InvalidDate", "31/02/2023", "02/31/2023", "InvalidTimestamp"), # Invalid dates
(4, None, None, None, None) # Null values   
]
# Define column names
columns = ["id", "date_iso", "date_dmy", "date_mdy", "timestamp"]

# Create DataFrame
df = spark.createDataFrame(data, schema=columns)

# Show the DataFrame
df.show(truncate=False)


+---+-----------+----------+----------+-------------------+
|id |date_iso   |date_dmy  |date_mdy  |timestamp          |
+---+-----------+----------+----------+-------------------+
|1  |2023-01-15 |15/01/2023|01/15/2023|2023-01-15 10:30:00|
|2  |2023-05-20 |20/05/2023|05/20/2023|2023-05-20 15:45:00|
|3  |InvalidDate|31/02/2023|02/31/2023|InvalidTimestamp   |
|4  |NULL       |NULL      |NULL      |NULL               |
+---+-----------+----------+----------+-------------------+



In [17]:
# DDL String for the schema
ddl_schema = """
id INT,
date_iso STRING,
date_dmy STRING,
date_mdy STRING,
timestamp STRING
"""
from pyspark. sql.types import StructType, StructField, IntegerType, StringType

# Read the CSV file into a DataFrame
df_file= spark.read.option("header", True).schema(ddl_schema).csv("/data/dates_data.csv")

# Show the DataFrame
df_file.show(truncate=False)

+---+-----------+-----------+----------+-------------------+
|id |date_iso   |date_dmy   |date_mdy  |timestamp          |
+---+-----------+-----------+----------+-------------------+
|1  |2023-01-15 |15/01/2023 |01/15/2023|2023-01-15 10:30:00|
|2  |2023-05-20 |20/05/2023 |05/20/2023|2023-05-20 15:45:00|
|3  |InvalidDate| 31/02/2023|02/31/2023| InvalidTimestamp  |
|4  |NULL       | ..        |NULL      |NULL               |
+---+-----------+-----------+----------+-------------------+



In [19]:
from pyspark.sql. functions import to_date

df = df\
.withColumn('parsed_date_iso', to_date(df.date_iso, 'yyyy-MM-dd' ) )\
.withColumn('parsed_date_dmy', to_date(df.date_dmy, 'dd/MM/yyyy'))\
.withColumn('parsed_date_mdy', to_date(df.date_mdy, 'MM/dd/yyyy'))

df.show(truncate=False)

+---+-----------+----------+----------+-------------------+---------------+---------------+---------------+
|id |date_iso   |date_dmy  |date_mdy  |timestamp          |parsed_date_iso|parsed_date_dmy|parsed_date_mdy|
+---+-----------+----------+----------+-------------------+---------------+---------------+---------------+
|1  |2023-01-15 |15/01/2023|01/15/2023|2023-01-15 10:30:00|2023-01-15     |2023-01-15     |2023-01-15     |
|2  |2023-05-20 |20/05/2023|05/20/2023|2023-05-20 15:45:00|2023-05-20     |2023-05-20     |2023-05-20     |
|3  |InvalidDate|31/02/2023|02/31/2023|InvalidTimestamp   |NULL           |NULL           |NULL           |
|4  |NULL       |NULL      |NULL      |NULL               |NULL           |NULL           |NULL           |
+---+-----------+----------+----------+-------------------+---------------+---------------+---------------+



In [20]:
# timestamp

In [23]:
from pyspark. sql. functions import to_timestamp, year, month, dayofmonth, hour, minute

df = df.withColumn('parsed_timestamp', to_timestamp(df.timestamp, 'yyyy-MM-dd HH: mm: ss' ) )
df. show()
df.printSchema()

+---+-----------+----------+----------+-------------------+---------------+---------------+---------------+----------------+
| id|   date_iso|  date_dmy|  date_mdy|          timestamp|parsed_date_iso|parsed_date_dmy|parsed_date_mdy|parsed_timestamp|
+---+-----------+----------+----------+-------------------+---------------+---------------+---------------+----------------+
|  1| 2023-01-15|15/01/2023|01/15/2023|2023-01-15 10:30:00|     2023-01-15|     2023-01-15|     2023-01-15|            NULL|
|  2| 2023-05-20|20/05/2023|05/20/2023|2023-05-20 15:45:00|     2023-05-20|     2023-05-20|     2023-05-20|            NULL|
|  3|InvalidDate|31/02/2023|02/31/2023|   InvalidTimestamp|           NULL|           NULL|           NULL|            NULL|
|  4|       NULL|      NULL|      NULL|               NULL|           NULL|           NULL|           NULL|            NULL|
+---+-----------+----------+----------+-------------------+---------------+---------------+---------------+----------------+


In [24]:
df = df\
.withColumn('year',year(df.parsed_timestamp))\
.withColumn('month',month(df.parsed_timestamp))\
.withColumn('day',dayofmonth(df.parsed_timestamp))\
.withColumn('hour',hour(df.parsed_timestamp))\
.withColumn('minute',minute(df.parsed_timestamp))
df.show()

+---+-----------+----------+----------+-------------------+---------------+---------------+---------------+----------------+----+-----+----+----+------+
| id|   date_iso|  date_dmy|  date_mdy|          timestamp|parsed_date_iso|parsed_date_dmy|parsed_date_mdy|parsed_timestamp|year|month| day|hour|minute|
+---+-----------+----------+----------+-------------------+---------------+---------------+---------------+----------------+----+-----+----+----+------+
|  1| 2023-01-15|15/01/2023|01/15/2023|2023-01-15 10:30:00|     2023-01-15|     2023-01-15|     2023-01-15|            NULL|NULL| NULL|NULL|NULL|  NULL|
|  2| 2023-05-20|20/05/2023|05/20/2023|2023-05-20 15:45:00|     2023-05-20|     2023-05-20|     2023-05-20|            NULL|NULL| NULL|NULL|NULL|  NULL|
|  3|InvalidDate|31/02/2023|02/31/2023|   InvalidTimestamp|           NULL|           NULL|           NULL|            NULL|NULL| NULL|NULL|NULL|  NULL|
|  4|       NULL|      NULL|      NULL|               NULL|           NULL|       

In [25]:
from pyspark. sql. functions import datediff

df = df.withColumn('days_dfference', datediff(df.parsed_date_mdy,df.parsed_date_iso))
df.select('parsed_date_mdy','parsed_date_iso','days_dfference').show(truncate=False)

+---------------+---------------+--------------+
|parsed_date_mdy|parsed_date_iso|days_dfference|
+---------------+---------------+--------------+
|2023-01-15     |2023-01-15     |0             |
|2023-05-20     |2023-05-20     |0             |
|NULL           |NULL           |NULL          |
|NULL           |NULL           |NULL          |
+---------------+---------------+--------------+



In [26]:
spark.stop()